In [13]:
%pip install -U langgraph typing_extensions

  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.16
    Uninstalling langgraph-1.2.16:
      Successfully uninstalled langgraph-1.2.16
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langgraph.graph import StateGraph,START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [2]:
# 1. Define the state
class CrashState(TypedDict):
  input:str
  step1:str
  step2:str
  step3:str

In [3]:
# 2. Define steps
def step_1(state:CrashState)->CrashState:
  print("Step1 executed")
  return {"step1":"done","input":state["input"]}

def step_2(state:CrashState) -> CrashState:
  print("Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
  time.sleep(30) # Simulate long-running hang
  return {"step2":"done"}

def step_3(state:CrashState)->CrashState:
  print("Step 3 executed")
  return {"step3":"done"}


In [9]:
# 3. Build the graph

builder = StateGraph(CrashState)
builder.add_node("step_1",step_1)
builder.add_node("step_2",step_2)
builder.add_node("step_3",step_3)

builder.add_edge(START,"step_1")
builder.add_edge('step_1',"step_2")
builder.add_edge('step_2',"step_3")
builder.add_edge('step_3',END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer,interrupt_before=['step_2'])

# 5. Configure thread ID

config = {
    "configurable": {
        "thread_id": "crash_1"
    }
}





In [12]:
# 6. Execute graph
try:
  print("Runnning graph: Please manually interrupt during Step 2 ....")
  graph.invoke({"input":"start"},config=config)
except KeyboardInterrupt:
  print("Kernel manually interrupt (crash simulated).")

Runnning graph: Please manually interrupt during Step 2 ....
Step1 executed


In [14]:
final_stage = graph.invoke(None,config=config)
print("\n Final State: ", final_stage)

Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
Step 3 executed

 Final State:  {'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}
